# JEE Dropout Prediction - ML Training Pipeline

This notebook demonstrates the complete ML training pipeline for JEE dropout prediction, including:
- Data loading and preprocessing
- SMOTE for class imbalance handling
- 5-fold stratified cross-validation
- XGBoost and RandomForest training
- Model selection and evaluation
- Artifact saving

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import warnings
from pathlib import Path

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
import joblib

warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## 2. Configuration

In [ ]:
# Paths
DATA_DIR = Path('../data')
ARTIFACTS_DIR = Path('../artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Dataset paths
DATASET2_PATH = DATA_DIR / 'jee_training_data.csv'

print(f"Data directory: {DATA_DIR}")
print(f"Artifacts directory: {ARTIFACTS_DIR}")

## 3. Load Dataset

In [ ]:
# Load primary dataset (Dataset 2 has more relevant features)
df = pd.read_csv(DATASET2_PATH)

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

## 4. Data Preprocessing

In [ ]:
# Check for missing values
print(f"Missing values: {df.isnull().sum().sum()}")
if df.isnull().sum().sum() > 0:
    print("Filling missing values with median...")
    df = df.fillna(df.median())

# Separate features and target
target_col = 'dropout'
X = df.drop(columns=[target_col])
y = df[target_col]

print(f"\nFeatures: {X.shape[1]}")
print(f"\nTarget distribution:")
print(f"- Class 0 (No Dropout): {sum(y == 0)} ({sum(y == 0)/len(y)*100:.1f}%)")
print(f"- Class 1 (Dropout): {sum(y == 1)} ({sum(y == 1)/len(y)*100:.1f}%)")

## 5. Save Feature Names

In [ ]:
feature_names = list(X.columns)

with open(ARTIFACTS_DIR / 'features.json', 'w') as f:
    json.dump(feature_names, f, indent=2)

print(f"Features saved: {len(feature_names)} features")
print(f"Feature names: {feature_names}")

## 6. Apply SMOTE for Class Balancing

In [ ]:
print("Applying SMOTE for class balancing...")
smote = SMOTE(random_state=42, k_neighbors=5)
X_resampled, y_resampled = smote.fit_resample(X, y)

print(f"Original distribution: Class 0: {sum(y == 0)}, Class 1: {sum(y == 1)}")
print(f"After SMOTE: Class 0: {sum(y_resampled == 0)}, Class 1: {sum(y_resampled == 1)}")

## 7. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

print(f"Train set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## 8. Feature Scaling

In [ ]:
print("Scaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete!")

## 9. Train XGBoost with 5-Fold CV

In [ ]:
print("Training XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)

# 5-fold CV for XGBoost
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
xgb_cv_scores = cross_val_score(xgb_model, X_train_scaled, y_train, cv=cv, scoring='roc_auc')
print(f"XGBoost 5-fold CV ROC-AUC: {xgb_cv_scores.mean():.4f} (+/- {xgb_cv_scores.std() * 2:.4f})")

# Train on full training set
xgb_model.fit(X_train_scaled, y_train)
print("XGBoost training complete!")

## 10. Train RandomForest with 5-Fold CV

In [ ]:
print("Training RandomForest...")
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

# 5-fold CV for RandomForest
rf_cv_scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=cv, scoring='roc_auc')
print(f"RandomForest 5-fold CV ROC-AUC: {rf_cv_scores.mean():.4f} (+/- {rf_cv_scores.std() * 2:.4f})")

# Train on full training set
rf_model.fit(X_train_scaled, y_train)
print("RandomForest training complete!")

## 11. Evaluate Models on Test Set

In [ ]:
# XGBoost predictions
xgb_pred = xgb_model.predict(X_test_scaled)
xgb_pred_proba = xgb_model.predict_proba(X_test_scaled)[:, 1]

xgb_accuracy = accuracy_score(y_test, xgb_pred)
xgb_precision = precision_score(y_test, xgb_pred)
xgb_recall = recall_score(y_test, xgb_pred)
xgb_f1 = f1_score(y_test, xgb_pred)
xgb_roc_auc = roc_auc_score(y_test, xgb_pred_proba)

print("XGBoost Test Metrics:")
print(f"- Accuracy: {xgb_accuracy:.4f}")
print(f"- Precision: {xgb_precision:.4f}")
print(f"- Recall: {xgb_recall:.4f}")
print(f"- F1 Score: {xgb_f1:.4f}")
print(f"- ROC-AUC: {xgb_roc_auc:.4f}")

# RandomForest predictions
rf_pred = rf_model.predict(X_test_scaled)
rf_pred_proba = rf_model.predict_proba(X_test_scaled)[:, 1]

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)
rf_roc_auc = roc_auc_score(y_test, rf_pred_proba)

print("\nRandomForest Test Metrics:")
print(f"- Accuracy: {rf_accuracy:.4f}")
print(f"- Precision: {rf_precision:.4f}")
print(f"- Recall: {rf_recall:.4f}")
print(f"- F1 Score: {rf_f1:.4f}")
print(f"- ROC-AUC: {rf_roc_auc:.4f}")

## 12. Select Best Model and Save

In [ ]:
# Select best model by ROC-AUC
if xgb_roc_auc >= rf_roc_auc:
    best_model = xgb_model
    best_model_name = "XGBoost"
    best_roc_auc = xgb_roc_auc
    print(f"Selected: XGBoost (ROC-AUC: {xgb_roc_auc:.4f})")
else:
    best_model = rf_model
    best_model_name = "RandomForest"
    best_roc_auc = rf_roc_auc
    print(f"Selected: RandomForest (ROC-AUC: {rf_roc_auc:.4f})")

# Save best model
model_path = ARTIFACTS_DIR / 'model.pkl'
joblib.dump(best_model, model_path)
print(f"Model saved to: {model_path}")

# Save scaler
scaler_path = ARTIFACTS_DIR / 'scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f"Scaler saved to: {scaler_path}")

## 13. Save Training Metadata

In [ ]:
metadata = {
    'model_type': best_model_name,
    'roc_auc': float(best_roc_auc),
    'features': feature_names,
    'n_features': len(feature_names),
    'n_samples_train': len(X_train),
    'n_samples_test': len(X_test),
    'xgb_cv_roc_auc_mean': float(xgb_cv_scores.mean()),
    'xgb_cv_roc_auc_std': float(xgb_cv_scores.std()),
    'rf_cv_roc_auc_mean': float(rf_cv_scores.mean()),
    'rf_cv_roc_auc_std': float(rf_cv_scores.std()),
    'test_metrics': {
        'accuracy': float(xgb_accuracy if best_model_name == 'XGBoost' else rf_accuracy),
        'precision': float(xgb_precision if best_model_name == 'XGBoost' else rf_precision),
        'recall': float(xgb_recall if best_model_name == 'XGBoost' else rf_recall),
        'f1': float(xgb_f1 if best_model_name == 'XGBoost' else rf_f1),
        'roc_auc': float(best_roc_auc)
    }
}

metadata_path = ARTIFACTS_DIR / 'metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata saved to: {metadata_path}")

## 14. Training Summary

In [ ]:
print("=" * 80)
print("Training Complete!")
print("=" * 80)
print(f"\nBest Model: {best_model_name}")
print(f"Test ROC-AUC: {best_roc_auc:.4f}")
print(f"\nArtifacts saved to: {ARTIFACTS_DIR}")
print("  - model.pkl")
print("  - scaler.pkl")
print("  - features.json")
print("  - metadata.json")
print("=" * 80)